In [1]:
%pip install sentencepiece pypdf2 pymupdf rouge-score bert-score
%pip install openai-whisper coqui-tts gtts

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import PyPDF2
def extract_text(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = " ".join([page.extract_text() for page in reader.pages])
    return text

loading English flan t5 model

In [ ]:
%pip install --upgrade "transformers>=4.37.0,<4.39.0"

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM

# --- ENGLISH FOUNDATION MODEL (FLAN-T5) ---
english_model_name = "google/flan-t5-base"
english_tokenizer = AutoTokenizer.from_pretrained(english_model_name)
english_model = AutoModelForSeq2SeqLM.from_pretrained(english_model_name)



loading tamil model

In [12]:

# --- TAMIL MODEL (LLaMA or Gemma 2B) ---
# Example: Tamil LLaMA (replace with correct HF model path)
tamil_model_name = "google/muril-base-cased"   # placeholder, update to actual
tamil_tokenizer = AutoTokenizer.from_pretrained(tamil_model_name)
tamil_model = AutoModelForCausalLM.from_pretrained(tamil_model_name, device_map="auto")



tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

If you want to use `BertLMHeadModel` as a standalone, add `is_decoder=True.`


model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

loading french model

In [ ]:
# --- FRENCH MODEL (CamemBERT) ---
french_model_name = "guillaumephd/t5-french-base"
french_tokenizer = AutoTokenizer.from_pretrained(french_model_name)
french_model = AutoModelForCausalLM.from_pretrained(french_model_name)

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

If you want to use `CamembertLMHeadModel` as a standalone, add `is_decoder=True.`


testing llm in there native languvage generation on prompt

In [7]:
def generate_answer(model, tokenizer, question, max_length=128):
    inputs = tokenizer(question, return_tensors="pt")
    outputs = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example usage
print("English:", generate_answer(english_model, english_tokenizer, "What is AI?"))



English: artificial intelligence


In [22]:
import sys
# sys.stdout.reconfigure(encoding='utf-8')  # Not needed in Jupyter


In [23]:
def generate_answer_t(model, tokenizer, question, max_length=128):
    inputs = tokenizer(question, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=max_length)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [24]:
print("Tamil:", generate_answer_t(tamil_model, tamil_tokenizer, "கணினி நுண்ணறிவு என்றால் என்ன?"))


Tamil: கணினி நுண்ணறிவு என்றால் என்ன?.........................................................................................................................


In [25]:
print("French:", generate_answer(french_model, french_tokenizer, "Quels sont les avantages de l'intelligence artificielle ?"))


/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `CamembertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


French: Quels sont les avantages de l'intelligence artificielle ? » » » » » » » » » » »


In [1]:
with open('english_text.md', 'r', encoding='utf-8') as f:
    english_text = f.read()

with open('tamil_text.md', 'r', encoding='utf-8') as f:
    tamil_text = f.read()

with open('french_text.md', 'r', encoding='utf-8') as f:
    french_text = f.read()

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    length_function=len
)
texts = [english_text, tamil_text, french_text]
# Chunk all texts in the 'texts' list
english_chunks = [text_splitter.split_text(text) for text in english_text]
tamil_chunks = [text_splitter.split_text(text) for text in tamil_text]
french_chunks = [text_splitter.split_text(text) for text in french_text]


In [36]:
%pip install whisper sounddevice numpy scipy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Preparing metadata (setup.py) ... done
  Created wheel for whisper: filename=whisper-1.1.10-py3-none-any.whl size=41167 sha256=bb959b834e613336e3a678e30c980ab683dcbc48efdcf64d1a6d91bb2dcc8fce
  Stored in directory: /home/kelvin/.cache/pip/wheels/34/b8/4e/9c4c3351d670e06746a340fb4b7d854c76517eec225e5b32b1
Successfully built whisper
Note: you may need to restart the kernel to use updated packages.


In [3]:
import whisper
model = whisper.load_model("base")  # or "small" for better accuracy
question_audio = "voice/Record (online-voice-recorder.com).mp3"
result = model.transcribe(question_audio)
question_text = result["text"]

In [4]:
print(question_text)

 Tell me about E.I.


In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# --- Embedding Models ---
embedding_models = {
    "en": HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
    "ta": HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"),
    "fr": HuggingFaceEmbeddings(model_name="dangvantuan/sentence-camembert-base")
}



In [ ]:
from langchain.vectorstores import Chroma

# --- Create Vector Stores ---
vector_stores = {}

# Optional: Define persist directory if you want to persist
persist_directory = "./chroma_db"

# Split each document into chunks
english_chunks = text_splitter.split_text(texts[0])
tamil_chunks = text_splitter.split_text(texts[1])
french_chunks = text_splitter.split_text(texts[2])

# English
vector_stores["en"] = Chroma.from_texts(
    texts=english_chunks,
    embedding=embedding_models["en"],
    collection_name="english_collection",
    persist_directory=persist_directory
)

# Tamil
vector_stores["ta"] = Chroma.from_texts(
    texts=tamil_chunks,
    embedding=embedding_models["ta"],
    collection_name="tamil_collection",
    persist_directory=persist_directory
)

# French
vector_stores["fr"] = Chroma.from_texts(
    texts=french_chunks,
    embedding=embedding_models["fr"],
    collection_name="french_collection",
    persist_directory=persist_directory
)


/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `CamembertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [10]:
from transformers import pipeline

# --- Models ---
# Models are loaded via HuggingFace pipeline below for each language
qa_models = {
    "en": pipeline(
        "text2text-generation",
        model="google/flan-t5-base",
        tokenizer="google/flan-t5-base"
    ),
    "ta": pipeline(
        "text-generation",
        model="google/muril-base-cased",  # Replace with actual Tamil model
        tokenizer="google/muril-base-cased"
    ),
    "fr": pipeline(
        "text-generation",
        model="guillaumephd/t5-french-base",
        tokenizer="guillaumephd/t5-french-base"
    )
}

2025-08-05 05:20:48.538046: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754371248.720029   16691 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754371248.771552   16691 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754371249.179674   16691 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754371249.179730   16691 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1754371249.179734   16691 computation_placer.cc:177] computation placer alr

ImportError: cannot import name 'GenerationMixin' from 'transformers.generation' (/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/transformers/generation/__init__.py)

In [ ]:
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline

# --- Retrievers ---
retrievers = {
    lang: vector_stores[lang].as_retriever(search_kwargs={"k": 10})
    for lang in ["en", "ta", "fr"]
}

# --- QA Chains ---
qa_chains = {}
for lang in ["en", "ta", "fr"]:
    llm = HuggingFacePipeline(pipeline=qa_models[lang])
    qa_chains[lang] = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retrievers[lang],
        return_source_documents=True
    )

In [13]:
def ask_question(question: str, language: str = "en"):
    # Step 1: Retrieve relevant chunks
    docs = retrievers[language].get_relevant_documents(question)
    
    # Step 2: Generate answer
    answer = qa_models[language](
        f"Question: {question}\nContext: {docs[0].page_content}",
        max_length=200
    )
    
    return {
        "answer": answer[0]["generated_text"],
        "source": docs[0].metadata.get("source", "")
    }



In [14]:
# Example Usage
french_question = """Le premier petit cochon a décidé d'aller vers le Sud. Alors qu'il
marchait le long de la route, il a rencontré un fermier qui portait une
botte de paille. Il lui a alors demandé poliment :"Pourriez-vous s'il vous
plaît me donner cette paille, que je puisse construire une maison?"""
print(ask_question(french_question, "fr"))
print(ask_question("பாரதியார் பற்றி சொல்லுங்கள்", "ta"))  # Tamil: "What is the capital of Paris?"
print(ask_question("What is Attention", "en"))  # 

/tmp/ipykernel_46622/2758288725.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  docs = retrievers[language].get_relevant_documents(question)
/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `CamembertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been s

{'answer': 'Question: Le premier petit cochon a décidé d\'aller vers le Sud. Alors qu\'il\nmarchait le long de la route, il a rencontré un fermier qui portait une\nbotte de paille. Il lui a alors demandé poliment :"Pourriez-vous s\'il vous\nplaît me donner cette paille, que je puisse construire une maison?\nContext: seek their fortunes. Section 2 Le pr emier petit cochon a décidé d\'aller vers le Sud. Alors qu\'il marchait le long de la r oute, il a r encontré un fermier qui portait une botte de paille. Il lui a alors demandé poliment : "Pourriez-vous s\'il vous plaît me donner cette paille, que je puisse construire une maison?" The first little pig decided to go south. As he walked along the road he met a farmer carrying a bundle of straw , so he asked the man politely: "Could you please give me that straw , so that I', 'source': ''}


/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Question: பாரதியார் பற்றி சொல்லுங்கள்\nContext: அனுப்பி விடும்படி சப் மாஜிஸ்டிரேட்டுக்கு உத்தரவு அனுப்பியிருக்கிறார் இது சாதாரண நடவடிக்கை முறைகளுக்கு முற்றிலும் விரோதமானது மேலும் அந்தப் பிராது இவரிடம் விசாரணைக்கு வந்து இரண்டு வாரங்களுக்கு மேலாகி விட்டது இவர் இதுவரை ஒன்றும் செய்யாமல் சும்மா இருந்து வருகிறார் இத்தோடு நிற்கவில்லை சி வ கம்பெனியார் பிராது விஷயமாக விசாரணைகள் நடத்தும்படி ஜில்லா போலீஸ் தலைவரிட மிருநீது உதீதரவு பெற்றிருந்த போலீஸ் இன்ஸ்பெக்டரை மிஸ்டர் வாலர் அழைத்து நீர் இந்த விஷயத்தில் யாதொன்றும் செய்ய வேண்டியதில்லை எல்லாம் நானே நடத்திக்????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????????', 'source': ''}


/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'active memory', 'source': ''}


In [15]:
print(ask_question(french_question, "fr"))
  # 

/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `CamembertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Question: Le premier petit cochon a décidé d\'aller vers le Sud. Alors qu\'il\nmarchait le long de la route, il a rencontré un fermier qui portait une\nbotte de paille. Il lui a alors demandé poliment :"Pourriez-vous s\'il vous\nplaît me donner cette paille, que je puisse construire une maison?\nContext: seek their fortunes. Section 2 Le pr emier petit cochon a décidé d\'aller vers le Sud. Alors qu\'il marchait le long de la r oute, il a r encontré un fermier qui portait une botte de paille. Il lui a alors demandé poliment : "Pourriez-vous s\'il vous plaît me donner cette paille, que je puisse construire une maison?" The first little pig decided to go south. As he walked along the road he met a farmer carrying a bundle of straw , so he asked the man politely: "Could you please give me that straw , so that I', 'source': ''}


In [16]:
print(ask_question("பாரதியார் பற்றி சொல்லுங்கள்", "ta"))  # Tamil: "What is the capital of Paris?"

/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'Question: பாரதியார் பற்றி சொல்லுங்கள்\nContext: அனுப்பி விடும்படி சப் மாஜிஸ்டிரேட்டுக்கு உத்தரவு அனுப்பியிருக்கிறார் இது சாதாரண நடவடிக்கை முறைகளுக்கு முற்றிலும் விரோதமானது மேலும் அந்தப் பிராது இவரிடம் விசாரணைக்கு வந்து இரண்டு வாரங்களுக்கு மேலாகி விட்டது இவர் இதுவரை ஒன்றும் செய்யாமல் சும்மா இருந்து வருகிறார் இத்தோடு நிற்கவில்லை சி வ கம்பெனியார் பிராது விஷயமாக விசாரணைகள் நடத்தும்படி ஜில்லா போலீஸ் தலைவரிட மிருநீது உதீதரவு பெற்றிருந்த போலீஸ் இன்ஸ்பெக்டரை மிஸ்டர் வாலர் அழைத்து நீர் இந்த விஷயத்தில் யாதொன்றும் செய்ய வேண்டியதில்லை எல்லாம் நானே நடத்திக்................................................................................................................................................................................................................................................................', 'source': ''}


In [17]:
print(ask_question("What is Attention", "en"))


/home/kelvin/miniconda3/envs/linux-deep-gpu/lib/python3.12/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': 'active memory', 'source': ''}


In [20]:
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from IPython.display import clear_output

# --- UI Components ---
language_dropdown = widgets.Dropdown(
    options=[("English", "en"), ("Tamil", "ta"), ("French", "fr")],
    value="en",
    description="Language:"
)

question_input = widgets.Textarea(
    value="",
    placeholder="Type your question here...",
    description="Question:",
    layout={"width": "80%"}
)

submit_button = widgets.Button(description="Ask", button_style="success")
output_area = widgets.Output()

# --- Styled Output Template ---
def styled_answer(answer: str, source: str, language: str):
    lang_colors = {"en": "#3498db", "ta": "#e74c3c", "fr": "#2ecc71"}
    return HTML(f"""
    <div style="
        border-left: 5px solid {lang_colors[language]};
        padding: 10px;
        margin: 10px 0;
        background: #f8f9fa;
        border-radius: 5px;
    ">
        <h4 style="color: {lang_colors[language]}; margin-top: 0;">Answer</h4>
        <p>{answer}</p>
        <hr style="margin: 5px 0;">
        <small><strong>Source:</strong> {source}</small>
    </div>
    """)

# --- Query Handler ---
def on_ask_button_clicked(b):
    with output_area:
        clear_output()
        display(widgets.HTML("<i>Searching documents...</i>"))
        
        # Show loading spinner
        spinner = widgets.HTML("""<div class="spinner-border text-primary" role="status">
                                <span class="sr-only">Loading...</span></div>""")
        display(spinner)
        
        try:
            result = ask_question(
                question=question_input.value,
                language=language_dropdown.value
            )
            clear_output()
            display(styled_answer(
                answer=result["answer"],
                source=result["source"],
                language=language_dropdown.value
            ))
        except Exception as e:
            clear_output()
            display(widgets.HTML(f"""<div class="alert alert-danger">
                                  <strong>Error:</strong> {str(e)}</div>"""))

submit_button.on_click(on_ask_button_clicked)

# --- Display UI ---
display(widgets.VBox([
    widgets.HBox([language_dropdown, question_input]),
    submit_button,
    output_area
]))

In [ ]:
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from IPython.display import clear_output

# --- UI Components ---
language_dropdown = widgets.Dropdown(
    options=[("English", "en"), ("Tamil", "ta"), ("French", "fr")],
    value="en",
    description="Language:"
)

question_input = widgets.Textarea(
    value="",
    placeholder="Type your question here...",
    description="Question:",
    layout={"width": "80%"}
)

submit_button = widgets.Button(description="Ask", button_style="success")
output_area = widgets.Output()

# --- Styled Output Template ---
def styled_answer(answer: str, source: str, language: str):
    lang_colors = {"en": "#3498db", "ta": "#e74c3c", "fr": "#2ecc71"}
    return HTML(f"""
    <div style="
        border-left: 5px solid {lang_colors[language]};
        padding: 10px;
        margin: 10px 0;
        background: #f8f9fa;
        border-radius: 5px;
    ">
        <h4 style="color: {lang_colors[language]}; margin-top: 0;">Answer</h4>
        <p>{answer}</p>
        <hr style="margin: 5px 0;">
        <small><strong>Source:</strong> {source}</small>
    </div>
    """)

# --- Query Handler ---
def on_ask_button_clicked(b):
    with output_area:
        clear_output()
        display(widgets.HTML("<i>Searching documents...</i>"))
        
        # Show loading spinner
        spinner = widgets.HTML("""<div class="spinner-border text-primary" role="status">
                                <span class="sr-only">Loading...</span></div>""")
        display(spinner)
        
        try:
            result = ask_question(
                question=question_input.value,
                language=language_dropdown.value
            )
            clear_output()
            display(styled_answer(
                answer=result["answer"],
                source=result["source"],
                language=language_dropdown.value
            ))
        except Exception as e:
            clear_output()
            display(widgets.HTML(f"""<div class="alert alert-danger">
                                  <strong>Error:</strong> {str(e)}</div>"""))

submit_button.on_click(on_ask_button_clicked)

# --- Display UI ---
display(widgets.VBox([
    widgets.HBox([language_dropdown, question_input]),
    submit_button,
    output_area
]))

In [ ]:
from IPython.display import display, HTML, Markdown
import ipywidgets as widgets
from IPython.display import clear_output

# --- UI Components ---
language_dropdown = widgets.Dropdown(
    options=[("English", "en"), ("Tamil", "ta"), ("French", "fr")],
    value="en",
    description="Language:"
)

question_input = widgets.Textarea(
    value="",
    placeholder="Type your question here...",
    description="Question:",
    layout={"width": "80%"}
)

submit_button = widgets.Button(description="Ask", button_style="success")
output_area = widgets.Output()

# --- Styled Output Template ---
def styled_answer(answer: str, source: str, language: str):
    lang_colors = {"en": "#3498db", "ta": "#e74c3c", "fr": "#2ecc71"}
    return HTML(f"""
    <div style="
        border-left: 5px solid {lang_colors[language]};
        padding: 10px;
        margin: 10px 0;
        background: #f8f9fa;
        border-radius: 5px;
    ">
        <h4 style="color: {lang_colors[language]}; margin-top: 0;">Answer</h4>
        <p>{answer}</p>
        <hr style="margin: 5px 0;">
        <small><strong>Source:</strong> {source}</small>
    </div>
    """)

# --- Query Handler ---
def on_ask_button_clicked(b):
    with output_area:
        clear_output()
        display(widgets.HTML("<i>Searching documents...</i>"))
        
        # Show loading spinner
        spinner = widgets.HTML("""<div class="spinner-border text-primary" role="status">
                                <span class="sr-only">Loading...</span></div>""")
        display(spinner)
        
        try:
            result = ask_question(
                question=question_input.value,
                language=language_dropdown.value
            )
            clear_output()
            display(styled_answer(
                answer=result["answer"],
                source=result["source"],
                language=language_dropdown.value
            ))
        except Exception as e:
            clear_output()
            display(widgets.HTML(f"""<div class="alert alert-danger">
                                  <strong>Error:</strong> {str(e)}</div>"""))

submit_button.on_click(on_ask_button_clicked)

# --- Display UI ---
display(widgets.VBox([
    widgets.HBox([language_dropdown, question_input]),
    submit_button,
    output_area
]))